# Análisis de anchura de cráneos egipcios

Este cuaderno guía el análisis descriptivo e inferencial solicitado para comparar la anchura de cráneos entre el periodo predinástico temprano (código 1) y el predinástico tardío (código 2). Incluye:

- Estadísticos de centralización, dispersión, asimetría y curtosis por submuestra.
- Diagramas de caja y bigotes.
- Contrastación de normalidad (Kolmogorov-Smirnov).
- Intervalos de confianza para la diferencia de medias en tres niveles.
- Test *t* para comparar las medias y discusión de sus supuestos.

Sigue las celdas de arriba a abajo y sustituye la ruta del archivo por la ubicación de tu Excel editable para trabajar con tus propios datos.

## Configuración de datos

- El archivo debe contener una columna con los códigos del periodo (`1` = predinástico temprano, `2` = predinástico tardío).
- Otra columna debe contener la anchura de los cráneos en milímetros.
- Ajusta la ruta del archivo, el nombre de la hoja y los nombres de columnas en la celda siguiente si tu estructura difiere.

Si el archivo no está disponible en la ruta indicada, se cargará un conjunto de datos de ejemplo meramente ilustrativo.

In [ ]:
import pathlib
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Ruta al archivo Excel editable con los datos reales
DATA_PATH = pathlib.Path('data/craneos.xlsx')  # actualiza según corresponda
SHEET_NAME = 0  # si tu Excel tiene varias hojas, ajusta el índice o el nombre
COLUMN_PERIOD = 'periodo'
COLUMN_WIDTH = 'anchura_mm'

np.random.seed(42)

if DATA_PATH.exists():
    df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
    # Renombrar columnas si es necesario para asegurar consistencia
    df = df.rename(columns={df.columns[0]: COLUMN_PERIOD, df.columns[1]: COLUMN_WIDTH})
    source_note = f'Datos cargados desde {DATA_PATH}'
else:
    # Conjunto de datos ilustrativo para que el cuaderno pueda ejecutarse sin el Excel original
    early = np.random.normal(loc=135, scale=4.5, size=32)
    late = np.random.normal(loc=138, scale=5.0, size=29)
    df = pd.DataFrame({
        COLUMN_PERIOD: np.concatenate([np.repeat(1, len(early)), np.repeat(2, len(late))]),
        COLUMN_WIDTH: np.concatenate([early, late]),
    })
    source_note = 'Datos de ejemplo generados aleatoriamente (reemplaza por tu Excel para el análisis real)'

# Depuración mínima para conservar solo registros válidos
clean_df = df[[COLUMN_PERIOD, COLUMN_WIDTH]].dropna()
clean_df[COLUMN_PERIOD] = clean_df[COLUMN_PERIOD].astype(int)

print(source_note)
clean_df.head()

## Estadísticos descriptivos por periodo

Se calculan medidas de centralización (media, mediana, moda), dispersión (desviación típica, varianza, rango intercuartílico), asimetría y curtosis para cada submuestra. También se muestra el número de observaciones para controlar el tamaño muestral.

In [ ]:
def compute_descriptives(series: pd.Series) -> pd.Series:
    mode_vals = series.mode()
    mode = mode_vals.iloc[0] if not mode_vals.empty else np.nan
    return pd.Series({
        'n': series.count(),
        'media': series.mean(),
        'mediana': series.median(),
        'moda': mode,
        'desviacion_tipica': series.std(ddof=1),
        'varianza': series.var(ddof=1),
        'q1': series.quantile(0.25),
        'q3': series.quantile(0.75),
        'rango_intercuartil': series.quantile(0.75) - series.quantile(0.25),
        'asimetria': stats.skew(series, bias=False),
        'curtosis': stats.kurtosis(series, fisher=True, bias=False),
    })

descriptive_table = (
    clean_df
    .groupby(COLUMN_PERIOD)[COLUMN_WIDTH]
    .apply(compute_descriptives)
    .unstack(0)
)

# Reordenar columnas para mayor legibilidad
ordered_columns = sorted(descriptive_table.columns)
descriptive_table = descriptive_table[ordered_columns]

descriptive_table

In [ ]:
display(Markdown(
    '**Comentario:** Observa las diferencias entre periodos.

'
    '- La media y mediana permiten comparar la tendencia central, mientras que el IQR y la desviación típica cuantifican la dispersión.
'
    '- Valores de asimetría cercanos a 0 sugieren simetría; la curtosis próxima a 0 se asemeja a la normal estándar.
'
    '- Utiliza estos valores para responder si las anchuras cambian entre periodos y si alguna distribución presenta colas largas o sesgo.'
))

## Diagramas de caja y bigotes

El diagrama resume medianas, cuartiles y posibles valores atípicos diferenciando cada periodo histórico.

In [ ]:
sns.set(style='whitegrid')
plt.figure(figsize=(8, 5))
ax = sns.boxplot(data=clean_df, x=COLUMN_PERIOD, y=COLUMN_WIDTH, palette='Set2')
ax.set_xlabel('Periodo (1 = predinástico temprano, 2 = predinástico tardío)')
ax.set_ylabel('Anchura de cráneo (mm)')
ax.set_title('Distribución de la anchura de cráneos por periodo')
plt.tight_layout()
plt.show()

## Test de normalidad (Kolmogorov-Smirnov)

El test KS compara la distribución empírica de cada submuestra con una normal teórica usando su media y desviación típica. Un *p*-valor grande (por ejemplo, > 0.05) indica que no hay evidencia fuerte contra la normalidad.

In [ ]:
def ks_test_normality(series: pd.Series) -> pd.Series:
    mean = series.mean()
    std = series.std(ddof=0)
    stat, pvalue = stats.kstest(series, 'norm', args=(mean, std))
    return pd.Series({'ks_stat': stat, 'pvalue': pvalue, 'media': mean, 'desv': std})

ks_results = clean_df.groupby(COLUMN_PERIOD)[COLUMN_WIDTH].apply(ks_test_normality).unstack(0)
ks_results

In [ ]:
for period, result in ks_results.items():
    conclusion = 'no se rechaza' if result['pvalue'] > 0.05 else 'se rechaza'
    display(Markdown(
        f"**Periodo {period}:** KS = {result['ks_stat']:.3f}, *p* = {result['pvalue']:.3f}. "
        f"Con α = 0.05 se {conclusion} la normalidad."
    ))


## Intervalos de confianza para la diferencia de medias

Se calcula la diferencia \(\mu_{1} - \mu_{2}\) usando el método de Welch (varianzas no supuestas iguales). Se incluyen los niveles de confianza 90%, 95% y 99%.

In [ ]:
def welch_se_and_df(series_a: pd.Series, series_b: pd.Series):
    var_a, var_b = series_a.var(ddof=1), series_b.var(ddof=1)
    n_a, n_b = series_a.count(), series_b.count()
    se = np.sqrt(var_a / n_a + var_b / n_b)
    df_num = (var_a / n_a + var_b / n_b) ** 2
    df_den = (var_a**2 / (n_a**2 * (n_a - 1))) + (var_b**2 / (n_b**2 * (n_b - 1)))
    df = df_num / df_den
    return se, df

grouped = clean_df.groupby(COLUMN_PERIOD)[COLUMN_WIDTH]
series1 = grouped.get_group(1)
series2 = grouped.get_group(2)

diff_means = series1.mean() - series2.mean()
se_diff, df_welch = welch_se_and_df(series1, series2)

confidence_levels = [0.90, 0.95, 0.99]
records = []
for level in confidence_levels:
    alpha = 1 - level
    t_crit = stats.t.ppf(1 - alpha / 2, df=df_welch)
    margin = t_crit * se_diff
    lower = diff_means - margin
    upper = diff_means + margin
    records.append({
        'nivel_confianza': level,
        'limite_inferior': lower,
        'limite_superior': upper,
        't_critico': t_crit,
        'grados_libertad': df_welch,
    })

ci_table = pd.DataFrame.from_records(records)
ci_table

In [ ]:
direction = 'mayor' if diff_means > 0 else 'menor'
period_wider = 1 if diff_means > 0 else 2
story = (
    f"La diferencia de medias (periodo 1 - periodo 2) es {diff_means:.2f} mm, lo que sugiere "
    f"que el periodo {period_wider} presenta anchuras {direction}es. "
    'Revisa si los intervalos contienen el 0: si no lo incluyen, la diferencia es compatible con ser distinta de cero al nivel indicado.'
)
display(Markdown(f"**Interpretación preliminar:** {story}"))

## Test *t* para la igualdad de medias

Se realiza el test *t* de Welch (no asume varianzas iguales). También se reporta el test de Levene para evaluar homocedasticidad. Recuerda que el test *t* requiere:

1. Independencia de las observaciones (asumida).
2. Normalidad aproximada en cada grupo o muestras grandes (ver test KS).
3. Igualdad de varianzas si se usa la versión clásica; el método de Welch relaja este supuesto.

In [ ]:
levene_stat, levene_p = stats.levene(series1, series2)
welch_t, welch_p = stats.ttest_ind(series1, series2, equal_var=False)
pooled_t, pooled_p = stats.ttest_ind(series1, series2, equal_var=True)

summary_rows = [
    {'prueba': 'Levene (varianzas iguales)', 'estadistico': levene_stat, 'pvalue': levene_p},
    {'prueba': 't de Welch (varianzas desiguales)', 'estadistico': welch_t, 'pvalue': welch_p},
    {'prueba': 't clásica (varianzas iguales)', 'estadistico': pooled_t, 'pvalue': pooled_p},
]
test_table = pd.DataFrame(summary_rows)
test_table

In [ ]:
levene_conclusion = 'se rechaza' if levene_p < 0.05 else 'no se rechaza'
welch_conclusion = 'diferencia significativa' if welch_p < 0.05 else 'sin evidencia de diferencia'
pooled_conclusion = 'diferencia significativa' if pooled_p < 0.05 else 'sin evidencia de diferencia'

text = (
    f"- **Levene:** p = {levene_p:.3f} → {levene_conclusion} la igualdad de varianzas a α = 0.05.
"
    f"- **t de Welch:** t = {welch_t:.2f}, p = {welch_p:.3f} → {welch_conclusion} entre medias.
"
    f"- **t clásica:** t = {pooled_t:.2f}, p = {pooled_p:.3f} → {pooled_conclusion} (interpretar solo si Levene sugiere varianzas similares)."
)
display(Markdown(text))